# Qwen grounding calibration: eight examples
Uses the existing Qwen/Qwen3-4B-Instruct-2507 checkpoint, not a raw pretrained base model. No Llama access approval is needed.
1. Let Drive sync. 2. Select T4 GPU. 3. Run all four code cells. 4. Send the final Latest report.
This is same-model evaluation because the diagnosis system also uses this Qwen checkpoint. It is not independent validation. Only eight synthetic calibration examples run; PHM diagnosis and explanation generation are not rerun.
The Qwen judge cache is separate from both Llama calibration and diagnostic caches. Cached attempts, including failures, are reused without automatic retry.


In [2]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Disconnect and delete runtime, reconnect, then complete Google authorisation. No inference has started.') from exc
if not (ROOT/'Files/phm_judge_calibration.py').is_file():
    raise FileNotFoundError(f'Wait for Drive sync or correct the project path: {ROOT}. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_judge_calibration.py': '48b0dc185b83f6fa0f0040599a6348234ebe69951969872d982931386d9e41e9', 'test_phm_judge_calibration.py': '2cf90544f0a3ae8ae1f4f5029cf34cabc9132f6bf53260acbcb5c66d3a0bf952', 'phm_grounding_review.py': 'bfd2dcc11ff8c8b63775f077a213bc743cf44f65b7af82211df2528fa7ae1140', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'requirements-qwen-colab.txt': '0290713affb7b64776db6d09da7735ef3abbf12c38cf511c473abb0ec76f4226', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b'}
PACK_HASHES = {'calibration_requests.jsonl': '9f0a53317e5ee720514e1519836d2528757c05d8e6960ce106f850b788ea0e54', 'calibration_answers_DO_NOT_SEND_TO_JUDGE.jsonl': '8e4110dd029a5db006792a61b3fb5c5a696f15cf814ab2b0f3cbb196927445ae'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256((ROOT/'Files'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
for name, expected in PACK_HASHES.items():
    assert hashlib.sha256((ROOT/'Outputs/reviews/grounding_v1'/name).read_bytes()).hexdigest() == expected, f'Not fully synced: {name}'
print('Code and calibration pack verified.')


Mounted at /content/drive
Code and calibration pack verified.


In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-qwen-colab.txt', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed. Using the existing public Qwen checkpoint; no Llama token/access check.')


Dependencies installed. Using the existing public Qwen checkpoint; no Llama token/access check.


In [4]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', 'test_phm_judge_calibration.py', '-v'], check=True)
subprocess.run([sys.executable, 'Files/phm_judge_calibration.py', '--judge', 'qwen', '--root', str(ROOT), '--prepare-only'], check=True)
print('Eight calibration requests prepared. Expected answers are excluded from model prompts.')


Eight calibration requests prepared. Expected answers are excluded from model prompts.


In [5]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun all cells.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'Files/phm_judge_calibration.py', '--judge', 'qwen', '--root', str(ROOT)], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('STOP: let Drive sync and send this report. Calibration only; no diagnosis or PHM explanation judging was run.')


GPU: Tesla T4
Latest report: {'run_id': '20261122T054354_Qwen_grounding_calibration_40fc079a', 'run_directory': 'Outputs/runs/20261122T054354_Qwen_grounding_calibration_40fc079a', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T05:50:24.320661+00:00'}
STOP: let Drive sync and send this report. Calibration only; no diagnosis or PHM explanation judging was run.
